In [ ]:
import pandas as pd
import numpy as np
import gc
import pickle
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, LSTM, Embedding, concatenate, Dropout

print("1. Loading and Preprocessing Data...")
# Load dataset
df = pd.read_csv('WELFake_Dataset.csv')
df = df.dropna(subset=['title', 'text'])

# Combine title and text first
df['full_text'] = df['title'].astype(str) + " " + df['text'].astype(str)

# Split by spaces, grab the first 500 words, and rejoin them safely
df['short_text'] = df['full_text'].apply(lambda x: " ".join(x.split()[:500]))

X_raw = df['short_text'].values
y = df['label'].values

print("2. Splitting Data...")
# Split raw text FIRST to avoid duplicating massive TF-IDF arrays in memory
X_train_text, X_test_text, y_train, y_test = train_test_split(
    X_raw, y, test_size=0.2, random_state=42
)

# Free up memory immediately
del df
del X_raw
gc.collect()

print("3. Building TF-IDF Features (Branch A)...")
max_tfidf_features = 5000 
# Force float32 to cut memory consumption in half
tfidf_vectorizer = TfidfVectorizer(max_features=max_tfidf_features, stop_words='english', dtype=np.float32)

X_train_tfidf = tfidf_vectorizer.fit_transform(X_train_text).toarray()
X_test_tfidf = tfidf_vectorizer.transform(X_test_text).toarray()
gc.collect()

print("4. Building Sequence Features (Branch B)...")
max_sequence_length = 500
max_words = 10000 

tokenizer = Tokenizer(num_words=max_words)
tokenizer.fit_on_texts(X_train_text) # Fit ONLY on training data

X_train_seq = pad_sequences(tokenizer.texts_to_sequences(X_train_text), maxlen=max_sequence_length)
X_test_seq = pad_sequences(tokenizer.texts_to_sequences(X_test_text), maxlen=max_sequence_length)

# Final memory cleanup before training
del X_train_text
del X_test_text
gc.collect()

print("5. Building Dual-Branch Keras Model...")
# --- Branch A: TF-IDF + Dense NN ---
input_tfidf = Input(shape=(max_tfidf_features,), name="tfidf_input")
dense_1 = Dense(128, activation='relu')(input_tfidf)
dropout_1 = Dropout(0.5)(dense_1)
dense_2 = Dense(64, activation='relu')(dropout_1)

# --- Branch B: Token Sequence + LSTM ---
input_seq = Input(shape=(max_sequence_length,), name="seq_input")
embedding = Embedding(input_dim=max_words, output_dim=128, input_length=max_sequence_length)(input_seq)
lstm_layer = LSTM(64)(embedding)

# --- Fusion Layer ---
merged = concatenate([dense_2, lstm_layer])
merged_dense = Dense(32, activation='relu')(merged)
merged_dropout = Dropout(0.5)(merged_dense)

# Output Layer
output_layer = Dense(1, activation='sigmoid', name="output")(merged_dropout)

model = Model(inputs=[input_tfidf, input_seq], outputs=output_layer)
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

print("6. Training Model...")
# Use batch_size=32 to prevent GPU Out-Of-Memory errors
history = model.fit(
    {"tfidf_input": X_train_tfidf, "seq_input": X_train_seq},
    y_train,
    validation_data=({"tfidf_input": X_test_tfidf, "seq_input": X_test_seq}, y_test),
    epochs=5,
    batch_size=32
)

print("7. Saving Model and Preprocessors...")
# Save everything needed for the FastAPI backend
model.save('dual_branch_fakenews_model.h5')

with open('tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf_vectorizer, f)

with open('tokenizer.pkl', 'wb') as f:
    pickle.dump(tokenizer, f)

print("Pipeline Complete! Download the .h5 and .pkl files from the Colab file explorer.")